# Per-feature-group AUROC (Supp S2)

AUROC by feature group (residual mean, std, RMS, max, skew, kurtosis, gravity-residual std/RMS) across four folds and three anomaly types.


**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).

**Input dependency.** This notebook is a T4-row extension. It reads the existing 3-fold AUROC values from a CSV produced by an earlier run, computes the T4 row, and merges. Required input:

- `Processed_Data/NB10f_feature_group_auroc.csv`

If the file is missing, the notebook raises an explicit error.


In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")

CANONICAL_CSV = os.path.join(OUT, "NB10f_feature_group_auroc.csv")
OUT_CSV       = os.path.join(OUT, "NB10f_feature_group_auroc_4fold.csv")

# Constants
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
PAYLOAD_COM  = np.array([0, 0, 0.05])
GRAVITY      = np.array([0, 0, -9.81])
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200

UR5_DH_A     = [0, -0.42500, -0.39225, 0, 0, 0]
UR5_DH_D     = [0.089159, 0, 0, 0.10915, 0.09465, 0.0823]
UR5_DH_ALPHA = [np.pi/2, 0, 0, np.pi/2, -np.pi/2, 0]
UR5_MASS     = [3.7000, 8.3930, 2.2750, 1.2190, 1.2190, 0.1879]
UR5_COM      = [
    [0.0,     -0.02561,  0.00193],
    [0.21250,  0.0,      0.11336],
    [0.11993,  0.0,      0.02650],
    [0.0,     -0.00180,  0.01634],
    [0.0,      0.00180,  0.01634],
    [0.0,      0.0,     -0.00116],
]

PSR_COLS = ([f"J{j}_{s}" for j in range(6)
             for s in ["resid_mean","resid_std","resid_rms","resid_max",
                       "resid_skew","resid_kurtosis",
                       "grav_resid_std","grav_resid_rms"]]
            + ["total_resid_rms","J1J2_resid_corr"])

FEATURE_GROUPS = {
    "gravity_residual":  [f"J{j}_{s}" for j in range(6)
                          for s in ["grav_resid_std", "grav_resid_rms"]],
    "residual_std":      [f"J{j}_resid_std"  for j in range(6)],
    "residual_mean":     [f"J{j}_resid_mean" for j in range(6)],
    "residual_rms":      [f"J{j}_resid_rms"  for j in range(6)],
    "residual_max":      [f"J{j}_resid_max"  for j in range(6)],
    "higher_moments":    [f"J{j}_{s}" for j in range(6)
                          for s in ["resid_skew", "resid_kurtosis"]],
    "cross_joint":       ["J1J2_resid_corr"],
    "total_rms":         ["total_resid_rms"],
}

REGISTRY = {
    "T1_healthy":   ("T1_PickPlace/Healthy",   "UR5_T1_healthy_180cyc_*.h5",        "T1","healthy","none"),
    "T2_healthy":   ("T2_Assembly/Healthy",    "UR5_T2_healthy_180cyc_*.h5",        "T2","healthy","none"),
    "T3_healthy":   ("T3_Palletize/Healthy",   "UR5_T3_healthy_183cyc_*.h5",        "T3","healthy","none"),
    "T4_healthy":   ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5","T4","healthy","none"),
    "T4_A2_0.5kg":  ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",       "T4","A2","0.5kg"),
    "T4_A2_1kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",         "T4","A2","1kg"),
    "T4_A2_2kg":    ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",         "T4","A2","2kg"),
    "T4_A3_7duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",      "T4","A3","7duct"),
    "T4_A3_14duct": ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",     "T4","A3","14duct"),
    "T4_A5_20mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",        "T4","A5","20mm"),
    "T4_A5_50mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",        "T4","A5","50mm"),
    "T4_A5_100mm":  ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",       "T4","A5","100mm"),
}

def dh_transform(a, d, alpha, theta):
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [0,    sa,    ca,    d   ],
        [0,    0,     0,     1   ]
    ])

def gravity_torque(q, payload_mass=0.0):
    T = [np.eye(4)]
    for i in range(6):
        T.append(T[-1] @ dh_transform(
            UR5_DH_A[i], UR5_DH_D[i], UR5_DH_ALPHA[i], q[i]))
    com_world = [(T[i+1] @ np.array([*UR5_COM[i], 1.0]))[:3] for i in range(6)]
    masses = list(UR5_MASS)
    if payload_mass > 0:
        masses.append(payload_mass)
        com_world.append((T[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3])
    tau_g = np.zeros(6)
    dq = 1e-6
    for i in range(6):
        qp = q.copy(); qp[i] += dq
        Tp = [np.eye(4)]
        for jj in range(6):
            Tp.append(Tp[-1] @ dh_transform(
                UR5_DH_A[jj], UR5_DH_D[jj], UR5_DH_ALPHA[jj], qp[jj]))
        for jj in range(len(masses)):
            cp = (Tp[jj+1] @ np.array([*UR5_COM[jj], 1.0]))[:3] if jj < 6 \
                 else (Tp[6] @ np.array([*PAYLOAD_COM, 1.0]))[:3]
            tau_g[i] += masses[jj] * GRAVITY @ ((cp - com_world[jj]) / dq)
    return tau_g

def fit_psr_fold(train_cycles):
    """OLS M4 weights with per-cycle payload."""
    rows = {j: [] for j in range(6)}
    for cyc in train_cycles:
        payload = TASK_PAYLOAD[cyc["task"]]
        q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
        N = len(q_a)
        for t in range(0, N, SUBSAMPLE):
            tau_g = gravity_torque(q_a[t], payload_mass=payload)
            for j in range(6):
                qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                         if 0 < t < N - 1 else 0.0)
                rows[j].append([tau_g[j], qd_a[t, j],
                                np.sign(qd_a[t, j]), qdd_j, 1.0,
                                cur[t, j]])
    psr_w = []
    for j in range(6):
        A = np.array(rows[j])
        w, _, _, _ = np.linalg.lstsq(A[:, :5], A[:, 5], rcond=None)
        psr_w.append(w)
    return psr_w

def extract_psr(cyc, psr_w):
    """50-dim PSR feature vector."""
    payload = TASK_PAYLOAD[cyc["task"]]
    q_a, qd_a, cur = cyc["q"], cyc["qd"], cyc["current"]
    N = len(q_a)
    idx = list(range(0, N, SUBSAMPLE))
    res = np.zeros((len(idx), 6))
    gr  = np.zeros((len(idx), 6))
    for ti, t in enumerate(idx):
        tau_g = gravity_torque(q_a[t], payload_mass=payload)
        for j in range(6):
            qdd_j = ((qd_a[t+1, j] - qd_a[t-1, j]) * RATE / 2.0
                     if 0 < t < N - 1 else 0.0)
            phi = np.array([tau_g[j], qd_a[t, j],
                            np.sign(qd_a[t, j]), qdd_j, 1.0])
            res[ti, j] = cur[t, j] - phi @ psr_w[j]
            gr[ti, j]  = cur[t, j] - (psr_w[j][0]*tau_g[j] + psr_w[j][4])
    f = {}
    for j in range(6):
        r = res[:, j]; g = gr[:, j]
        f[f"J{j}_resid_mean"]     = r.mean()
        f[f"J{j}_resid_std"]      = r.std()
        f[f"J{j}_resid_rms"]      = np.sqrt(np.mean(r**2))
        f[f"J{j}_resid_max"]      = np.abs(r).max()
        f[f"J{j}_resid_skew"]     = float(sst.skew(r))
        f[f"J{j}_resid_kurtosis"] = float(sst.kurtosis(r))
        f[f"J{j}_grav_resid_std"] = g.std()
        f[f"J{j}_grav_resid_rms"] = np.sqrt(np.mean(g**2))
    f["total_resid_rms"] = np.sqrt(np.mean(res**2))
    f["J1J2_resid_corr"] = float(np.corrcoef(res[:, 1], res[:, 2])[0, 1]
                                  if len(res) > 2 else 0.0)
    return np.array([f[k] for k in PSR_COLS])

# Main
print("=" * 70)
print("NB10f_T4_extension -- per-feature-group AUROC T4 extension")
print("=" * 70)

# Step 1: load 3-fold CSV
print(f"\n[Step 1] Loading {os.path.basename(CANONICAL_CSV)}...")
if not os.path.exists(CANONICAL_CSV):
    raise RuntimeError(
        f"Canonical CSV not found at {CANONICAL_CSV}. "
        "This is the source of the published Supp S3 T1/T2/T3 values."
    )
canon = pd.read_csv(CANONICAL_CSV)
print(f"  Loaded {len(canon)} canonical rows.")
print(f"  Columns: {list(canon.columns)}")
print(f"  Tasks: {sorted(canon['test_task'].unique())}")
print(f"  Anomalies: {sorted(canon['anomaly_type'].unique())}")
print(f"  Groups: {sorted(canon['group'].unique())}")

# Step 2: load data
print("\n[Step 2] Loading HDF5 data...")
all_cycles = []
for key, (subdir, pattern, task, anomaly, severity) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING  Not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        cur_all = f["actual_current"][:]
        q_all   = f["actual_q"][:]
        qd_all  = f["actual_qd"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "q": q_all[mask], "qd": qd_all[mask],
                "current": cur_all[mask], "task": task,
                "anomaly": anomaly, "severity": severity,
                "is_anomaly": is_anom,
            })
print(f"  Total cycles: {len(all_cycles)}")

# Step 3: PSR fit on T1+T2+T3 healthy
print("\n[Step 3] Fitting PSR on T1+T2+T3 healthy...")
tr_healthy = [c for c in all_cycles
              if c["task"] in ["T1", "T2", "T3"] and c["is_anomaly"] == 0]
te_cycles  = [c for c in all_cycles if c["task"] == "T4"]
print(f"  Train (T1+T2+T3 healthy): {len(tr_healthy)} cycles")
print(f"  Test (T4): {len(te_cycles)} cycles")
t0 = time.perf_counter()
psr_w = fit_psr_fold(tr_healthy)
print(f"  PSR fit time: {time.perf_counter()-t0:.1f}s")

# Step 4: extract features
print("\n[Step 4] Extracting PSR features...")
t0 = time.perf_counter()
Xtr = np.array([extract_psr(c, psr_w) for c in tr_healthy])
Xte = np.array([extract_psr(c, psr_w) for c in te_cycles])
print(f"  Feature extraction: {time.perf_counter()-t0:.1f}s")
print(f"  Xtr shape: {Xtr.shape}; Xte shape: {Xte.shape}")

# Step 5: per-feature-group AUROC for T4
print("\n[Step 5] Computing per-feature-group AUROC at T4...")
mu_tr = Xtr.mean(0); sg_tr = Xtr.std(0) + 1e-8
Xte_z = np.abs((Xte - mu_tr) / sg_tr)

te_h_idx = [i for i, c in enumerate(te_cycles) if c["is_anomaly"] == 0]
t4_rows = []

for anom in ["A2", "A3", "A5"]:
    anom_idx = [i for i, c in enumerate(te_cycles) if c["anomaly"] == anom]
    if not anom_idx:
        print(f"  WARNING: no T4 {anom} cycles")
        continue
    comb_idx = te_h_idx + anom_idx
    y_anom   = np.array([te_cycles[i]["is_anomaly"] for i in comb_idx])

    for group_name, group_feats in FEATURE_GROUPS.items():
        fi_list = [PSR_COLS.index(f) for f in group_feats if f in PSR_COLS]
        if not fi_list:
            continue
        sc_group = Xte_z[comb_idx][:, fi_list].mean(1)
        try:
            auroc = roc_auc_score(y_anom, sc_group)
        except ValueError:
            auroc = np.nan
        t4_rows.append(dict(
            test_task="T4", anomaly_type=anom,
            group=group_name, n_features=len(fi_list),
            auroc=round(float(auroc), 4)))
        print(f"  T4 | {anom} | {group_name:20s}: AUROC={auroc:.4f}")

# Step 6: merge 3-fold rows with T4 row, save
print(f"\n[Step 6] Merging with canonical CSV...")
combined = pd.concat([canon, pd.DataFrame(t4_rows)], ignore_index=True)
combined = combined.sort_values(["anomaly_type", "test_task", "auroc"],
                                 ascending=[True, True, False]).reset_index(drop=True)
combined.to_csv(OUT_CSV, index=False)
print(f"  Saved: {OUT_CSV} ({len(combined)} rows)")

# Step 7: integrity check
print(f"\n[Integrity check] T1/T2/T3 rows preserved byte-for-byte:")
match = True
for _, canon_row in canon.iterrows():
    sel = combined[(combined["test_task"] == canon_row["test_task"]) &
                   (combined["anomaly_type"] == canon_row["anomaly_type"]) &
                   (combined["group"] == canon_row["group"])]
    if len(sel) != 1 or float(sel.iloc[0]["auroc"]) != float(canon_row["auroc"]):
        match = False
        print(f"  DRIFT: {canon_row['test_task']} {canon_row['anomaly_type']} "
              f"{canon_row['group']}: canon={canon_row['auroc']} "
              f"new={sel.iloc[0]['auroc']}")
if match:
    print(f"  All {len(canon)} canonical rows preserved exactly.")

# Step 8: print revised Supp S3 (mean across 4 tasks)
print("\n" + "=" * 90)
print("REVISED SUPP TABLE S3 (mean across 4 LOTO tasks)")
print("=" * 90)
print(f"  {'Feature group':22s}  {'n_feat':6}  {'A2':8}  {'A3':8}  {'A5':8}  {'Mean':8}")
print("  " + "-" * 70)

# Display name mapping
DISPLAY = {
    "gravity_residual": "Gravity residual",
    "total_rms":        "Total RMS",
    "higher_moments":   "Higher moments",
    "residual_max":     "Residual (max)",
    "residual_mean":    "Residual (mean)",
    "residual_rms":     "Residual RMS",
    "cross_joint":      "Cross joint",
    "residual_std":     "Residual STD",
}

summary_rows = []
for group_name in FEATURE_GROUPS:
    sub = combined[combined["group"] == group_name]
    nf  = len(FEATURE_GROUPS[group_name])
    means = []
    cells = []
    for anom in ["A2", "A3", "A5"]:
        vals = sub[sub["anomaly_type"] == anom]["auroc"].values
        m = float(np.mean(vals))
        means.append(m)
        cells.append(f"{m:.3f}")
    overall_mean = float(np.mean(means))
    summary_rows.append({
        "Feature group": DISPLAY[group_name],
        "n features":    nf,
        "A2 (added mass)": cells[0],
        "A3 (friction)":   cells[1],
        "A5 (TCP offset)": cells[2],
        "Mean AUROC":      f"{overall_mean:.3f}",
        "_mean_sort":      overall_mean,
    })

# Sort by mean AUROC descending
summary_rows.sort(key=lambda r: -r["_mean_sort"])

for r in summary_rows:
    print(f"  {r['Feature group']:22s}  {r['n features']:6}  "
          f"{r['A2 (added mass)']:8}  {r['A3 (friction)']:8}  "
          f"{r['A5 (TCP offset)']:8}  {r['Mean AUROC']:8}")
print("=" * 90)

# Save summary as Supp S3 wide-format CSV
summary_df = pd.DataFrame([{k: r[k] for k in
    ["Feature group", "n features", "A2 (added mass)",
     "A3 (friction)", "A5 (TCP offset)", "Mean AUROC"]} for r in summary_rows])
SUMMARY_PATH = os.path.join(OUT, "NB10f_supp_s3_4fold_table.csv")
summary_df.to_csv(SUMMARY_PATH, index=False)
print(f"\nSupp S3 wide-format saved: {SUMMARY_PATH}")

print("\nNB10f_T4_extension complete.")